# EMS evaluation — start here
Run all cells to test the pipeline while the doctor annotates. The default is **MOCK: artificial software tests, not clinical results**. No API calls, credentials, or app server needed.

We evaluate **patient awareness, changes, preparation category, and optional queue ordering**. Ranking is not the only target. The clinician remains responsible for clinical decisions.

This notebook uses the ten fictional reports already sent to the doctor. They are text fixtures, not real EMT audio. For installation, file contracts, approval steps, and metric definitions, see [`evals/README.md`](../evals/README.md).

In [ ]:
from pathlib import Path
import sys
from datetime import datetime, timezone

ROOT = Path.cwd().resolve()
if not (ROOT / 'evals').is_dir() and (ROOT.parent / 'evals').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'evals').is_dir(), 'Start Jupyter from this repository or its analytics folder.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

MODE = 'mock'  # Change to 'clinician' only after reviewing reference answers.
REFERENCES_FILE = ROOT / 'evals/local/approved-references.json'
PREDICTIONS_FILE = ROOT / 'evals/local/model-predictions.json'
REVIEWS_FILE = None  # Optional: ROOT / 'evals/local/summary-reviews.json'
SAVE_REPORT = False  # True writes a new timestamped local report directory.
assert MODE in ('mock', 'clinician')


In [ ]:
from evals.contracts import load_cases, load_json
from evals.demo import mock_data
from evals.evaluate import evaluate
from evals.report import comparison_rows, render_report, save_report, table

cases = load_cases(ROOT / 'docs/clinician-annotation/pilot_cases.json')
queues = load_json(ROOT / 'evals/queues.json')
print(f'{len(cases)} fictional cases; {len(queues)} optional queue snapshots.')
print('No independent patient outcomes, source audio, or complete transport trajectories.')


## Model comparison architecture — plan only
The next cell previews configured API/local candidates and two prompt variants. It does **not** contact a provider, query a local server, install software, or download a model. See [`evals/MODEL_COMPARISON.md`](../evals/MODEL_COMPARISON.md) for the open-weight shortlist and explicit opt-in CLI workflow.

Prompt tuning uses development cases; a final comparison needs a separate, frozen clinician-reviewed test set. Schema/latency results alone cannot tell us which model is clinically better.

In [ ]:
from collections import Counter
from IPython.display import HTML, display
from evals.experiments import build_plan

experiment_spec = load_json(ROOT / 'evals/experiments.json')
plan, planned_jobs, _, _ = build_plan(experiment_spec, cases, queues)
counts = Counter(job['model']['id'] for job in planned_jobs)
display(HTML(table(['Candidate', 'Provider', 'Model', 'Planned requests'],
    [[m['id'], m['provider'], m['model'], counts[m['id']]] for m in experiment_spec['models']])))
print(f"PLAN ONLY: {plan['planned_requests']} requests. Nothing executed.")
print('No best-model result yet: real outputs, approved references, and summary review are required.')


## When answers arrive
1. Download the **current** Google Sheet as `.xlsx` and keep it under `evals/local/`. The committed workbook is only an older snapshot.
2. Run `python -m evals import-annotations --workbook evals/local/doctor-export.xlsx --out evals/local/import-01` from the repo root.
3. Review the imported JSON, preserve the original, and save accepted references as `approved-references.json`. Fill a pseudonymous `reviewer_id`, freeze `rubric_version`, and set `approved: true` only after checking complete answers. **Done is not approval.**
4. Save model outputs in the prediction contract documented in the README; set `MODE = 'clinician'` and the file paths above, then rerun all cells. Never put doctor answers in model inputs.

Partial answers remain visible, but unapproved cases are not scored. No summary grades means summary quality is N/A. This notebook never edits the doctor's workbook.

In [ ]:
if MODE == 'mock':
    references, predictions, reviews = mock_data(cases, queues)
    print('MOCK ONLY: reference labels, candidate outputs, telemetry, and grades are artificial.')
else:
    references = load_json(REFERENCES_FILE)
    predictions = load_json(PREDICTIONS_FILE)
    reviews = load_json(REVIEWS_FILE) if REVIEWS_FILE is not None else None

report = evaluate(cases, references, predictions, queues, reviews, allow_mock=MODE == 'mock')
print(report['mode'])
print(f"Approved references: {report['approved_reference_cases']}/{len(cases)}")
if not report['approved_reference_cases']:
    print('Waiting for approved case references: case quality scores are N/A. Queue approval is separate.')


In [ ]:
from IPython.display import HTML, display

rows = comparison_rows(report)
columns = ['config_id', 'change_precision', 'change_recall', 'change_f1',
           'preparation_accuracy', 'preparation_macro_f1',
           'prepare_now_false_negative_rate', 'pairwise_accuracy',
           'usable_response_rate', 'content_recall', 'factual_precision']
display(HTML('<h3>' + report['mode'] + '</h3>' + table(columns, [[r[c] for c in columns] for r in rows])))
print('Rates use 0–1. N/A means unavailable, not zero. Repeats are not independent patients.')


In [ ]:
# Full dashboard: confusion matrices, reliability, cost, and expandable case review.
display(HTML(render_report(report)))


## Read the results honestly
- **Precision / Recall / F1:** identify meaningful change; clinician `Unclear` and `No earlier report` are not negative labels.
- **Accuracy / Macro-F1 / Confusion Matrix:** agreement on preparation, not proof of a validated triage system.
- **False Negative Rate:** how often a reference `Prepare now` is missed, including failures/abstentions.
- **Pairwise Accuracy:** agreement on strictly ordered patient pairs; reference ties and uncertainty are handled separately.
- **Content Recall / Factual Precision:** require separately approved human summary grades. No automated word-overlap substitute.
- **Latency / Cost / Error Rate:** supplied telemetry and expected attempts; failed outputs are never quietly dropped.

These cases cannot yet establish AUROC, calibration, lead time, actual ED readiness, rural benefit, or time saved. Do not tune on these ten cases and report them as a held-out test set.

In [ ]:
if SAVE_REPORT:
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    output_dir = ROOT / 'analytics/results' / f'{MODE}-{stamp}'
    save_report(report, output_dir)
    print(f'Saved HTML dashboard, metrics JSON, and SAS-friendly CSVs: {output_dir}')
else:
    print('Nothing written. Set SAVE_REPORT = True to export a new local run.')


In [ ]:
if MODE == 'mock':
    perfect, flawed, failed = report['results']
    assert perfect['preparation']['accuracy'] == 1
    assert perfect['queue']['pairwise_accuracy'] == 1
    assert flawed['preparation']['accuracy'] < 1
    assert failed['usable_response_rate'] == 0
    assert failed['schema_validation_rate'] is None
    print('Mock smoke checks passed. These assertions verify code, not medicine.')


## Next experiments
Freeze the doctor-approved rubric → collect matched model/prompt runs → review summaries blinded to candidate → obtain a separate test set.

For product utility, build a **separate longitudinal replay study** comparing one-time prenotification with continuous updates on identical trajectories. Measure warning time, correct prioritization time, missed changes, false alerts, and queue churn. Vary missing/delayed information and transport length without confounding these with clinical severity. Such results demonstrate simulation behavior, not improved patient outcomes.